# Staged-to-real fall detection

Do fall detectors trained on **staged lab falls** work on **real falls**? Run the cells in order.
Everything is saved to Google Drive in `fall-cvpr/`. If Colab disconnects, run **Setup** again and continue at the step you were on; finished work is never redone.

**First:** *Runtime → Change runtime type → T4 GPU → Save*.

| Step | What it does | GPU | Time |
|---|---|---|---|
| Setup | connects Drive, downloads code | – | ~1 min |
| 1a. Quick test | processes 200 real clips to check everything works | yes | ~10 min |
| 1b. Full run | the rest of the ~11k clips, with two AI models (CLIP image model + VideoMAE video model); keeps ~3,600 short clips for labelling | yes | about 1–3 hours (speed is printed) |
| 2. Label | you mark clips: fall / not a fall / unsure | no | ~5–10 s per clip |
| 3. Staged data | downloads and encodes the UR Fall lab dataset | yes | ~5 min |
| 4. Results | trains, evaluates, shows the table and chart | no | ~1 min |

## Setup

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (fine for step 2/4; steps 1 and 3 need a T4 GPU)")
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fall-cvpr'
!mkdir -p {OUT}
!rm -rf /content/fall-detector && git clone -q -b claude/beautiful-brown-gdjzcq https://github.com/apersonli55555/fall-detector.git /content/fall-detector
!pip -q install open_clip_torch av
import sys; sys.path.insert(0, '/content/fall-detector/vision')
print("ready")

## Step 1a: quick test (about 10 minutes)
Processes the first 200 real clips. If it ends with `done: ...` and no errors, go on to 1b; the 200 already done are kept.

In [ ]:
!python /content/fall-detector/vision/oops_mine.py --out {OUT}/oops --limit 200

## Step 1b: full run (about 1–3 hours)
Keep this browser tab open. If the download drops, it reconnects by itself. If Colab disconnects, run **Setup** and this cell again: it skips what's done.

In [ ]:
!python /content/fall-detector/vision/oops_mine.py --out {OUT}/oops

## Step 2: label clips (no GPU needed)
To save GPU time: *Runtime → Change runtime type → CPU*, then run **Setup** again.

**Fall** = a **person** (not an animal) unintentionally ends up on the ground, floor or a lower level, in this clip.
Use *Unsure* if you honestly can't tell.

**Keyboard:** click once inside the labelling area, then press **F** (fall), **N** (not a fall), **U** (unsure), **Z** (undo).

The **test-set clips come first**: label all of them (the counter shows progress). Then label as many training-pool clips as you like; a few hundred is a good start.
Every press is saved immediately, so stop whenever you want.

In [ ]:
from labeler import Labeler
Labeler(f'{OUT}/oops').show()

## Step 3: staged lab falls (UR Fall dataset; GPU)

In [ ]:
!python /content/fall-detector/vision/urfall_prep.py --out {OUT}/urfall.npz --cache /content/urfall

## Step 4: results
Shows the table and the ROC chart. **Send me the table** (copy the text output), and I'll interpret it and plan the next step.

In [ ]:
!python /content/fall-detector/vision/experiment.py --staged {OUT}/urfall.npz --oops {OUT}/oops
from IPython.display import Image, display
display(Image(f'{OUT}/oops/results/roc_real.png', width=450))

## Step 5: real vs. synthetic vs. staged (GPU, about 20–40 min; no labelling)
Downloads OmniFall's synthetic fall videos (OF-Syn, 9.7 GB, streamed), encodes about 5,000 clips with the same two models, then re-runs the experiment with an equal-budget comparison: how many synthetic falls is one real fall worth?

In [ ]:
!pip -q install open_clip_torch av
!python /content/fall-detector/vision/ofsyn_prep.py --out {OUT}/ofsyn.npz
!python /content/fall-detector/vision/experiment.py --staged {OUT}/urfall.npz --oops {OUT}/oops --synthetic {OUT}/ofsyn.npz
from IPython.display import Image, display
display(Image(f'{OUT}/oops/results/equal_budget.png', width=480))

## Step 6: external real-world test on HMDB51 (GPU, about 20–30 min; no labelling)
HMDB51 (CC BY 4.0) is independent of OOPS!. Its human-labelled "fall_floor" clips are real-world falls; 20 clips from each other class are negatives. Nothing is trained on it, so this checks whether "real beats synthetic" holds outside OOPS!.

In [ ]:
!pip -q install open_clip_torch av pyarrow
!python /content/fall-detector/vision/hmdb_prep.py --out {OUT}/hmdb.npz --cache /content/hmdb
!python /content/fall-detector/vision/experiment.py --staged {OUT}/urfall.npz --oops {OUT}/oops --synthetic {OUT}/ofsyn.npz --external {OUT}/hmdb.npz

## Step 7: fine-tune VideoMAE (GPU, about 1.5–2.5 hours; no labelling)
Pre-registered test: does fine-tuning on synthetic + real data beat the best frozen probe on both FallOrNot and HMDB51? Every stage skips work that is already done, so if Colab disconnects, run Setup and this cell again.

In [ ]:
!pip -q install open_clip_torch av pyarrow
W = f'{OUT}/finetune'
!python /content/fall-detector/vision/finetune.py prepare-oops --work {W} --oops {OUT}/oops
!python /content/fall-detector/vision/finetune.py prepare-syn --work {W}
for tag in ['synthetic+real', 'synthetic', 'real']:
    !python /content/fall-detector/vision/finetune.py train --work {W} --train-on {tag}
    !python /content/fall-detector/vision/finetune.py score --work {W} --train-on {tag} --hmdb {OUT}/hmdb.npz --hmdb-cache /content/hmdb
!python /content/fall-detector/vision/experiment.py --staged {OUT}/urfall.npz --oops {OUT}/oops --synthetic {OUT}/ofsyn.npz --external {OUT}/hmdb.npz --finetuned {W}/scores_synthetic_real.npz {W}/scores_synthetic.npz {W}/scores_real.npz